In [28]:
import re
import numpy as np
import pandas as pd

In [29]:
dtype_map = {
    "Task Id": "string",
    "FPOGV": "Int64", "FPOGID": "Int64",
    "LPMMV": "Int64", "RPMMV": "Int64",
    "BKID": "Int64",  "BKDUR": "float64", 
    "AOI": "string"
}

fix_df = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/fix_labeled_with_blocks.csv", dtype=dtype_map, low_memory=False)
gaze_df = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/gaze_labeled_with_blocks.csv", dtype=dtype_map, low_memory=False)

In [30]:
def keep_on_off(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return only rows where df['label'] is 'ON' or 'OFF'.
    No phase-based mapping—just strict label filtering.
    """
    if "label" not in df.columns:
        raise KeyError("DataFrame has no 'label' column.")

    out = df.copy()
    out["label"] = out["label"].astype(str).str.strip().str.upper()
    out = out[out["label"].isin(["ON", "OFF"])].copy()

    # quick sanity check
    print("Kept rows:", len(out))
    print(out["label"].value_counts())

    return out

# Example usage
fixations_onoff = keep_on_off(fixations_labeled)
gaze_onoff      = keep_on_off(gaze_labeled)



fixations_onoff.to_csv("outputs/fixations_ON_OFF_only.csv", index=False)
gaze_onoff.to_csv("outputs/gaze_ON_OFF_only.csv", index=False)

Kept rows: 40862
label
ON     28295
OFF    12567
Name: count, dtype: int64
Kept rows: 1212581
label
ON     808805
OFF    403776
Name: count, dtype: int64


In [31]:
TICK_HZ = 10_000_000  # TIMETICK(f=10000000)

# --- Build ONE shared anchor per (Video, User) across any number of tables ---
def build_shared_anchor(*dfs,
                        tick_col: str = "TIMETICK(f=10000000)",
                        group_cols=("Video Name (folder)", "User ID (file)")) -> pd.DataFrame:
    refs = []
    for d in dfs:
        tmp = d[list(group_cols) + [tick_col]].copy()
        tmp[tick_col] = pd.to_numeric(tmp[tick_col], errors="coerce")
        refs.append(tmp.groupby(list(group_cols), as_index=False)[tick_col].min()
                        .rename(columns={tick_col: "min_tick"}))
    ref = (pd.concat(refs, ignore_index=True)
             .groupby(list(group_cols), as_index=False)["min_tick"].min()
             .rename(columns={"min_tick": "anchor_tick"}))
    return ref

# 1) Window ids by TIMETICK (per Video+User), no task-start anchoring
def add_window_id_by_timetick(
    df: pd.DataFrame,
    *,
    win_sec,
    tick_col: str = "TIMETICK(f=10000000)",
    group_cols=("Video Name (folder)", "User ID (file)"),
    shared_ref: pd.DataFrame | None = None,   # << pass build_shared_anchor(...) here for perfect alignment
    out_col: str = "win_id",
) -> pd.DataFrame:
    out = df.copy()
    out[tick_col] = pd.to_numeric(out[tick_col], errors="coerce")
    win_ticks = int(round(win_sec * TICK_HZ))

    if shared_ref is not None:
        # use the SAME anchor_tick for both streams
        out = out.merge(shared_ref, on=list(group_cols), how="left", validate="many_to_one")
        anchor = out["anchor_tick"]
    else:
        # fallback: per-DF min (works, but can shift if streams start at different ticks)
        anchor = out.groupby(list(group_cols))[tick_col].transform("min")

    out[out_col] = ((out[tick_col] - anchor) // win_ticks).astype("Int64")
    return out

In [32]:
# group keys WITHOUT 'label'
BASE_KEYS = ["Video Name (folder)","User ID (file)","Task Id","type","phase","block_id","win_id"]

def window_features_fixations(fix_df: pd.DataFrame) -> pd.DataFrame:
    eps = 1e-6
    def _agg(w: pd.DataFrame) -> pd.Series:
        fix_count     = w["FPOGID"].nunique() if "FPOGID" in w else np.nan
        fix_mean_dur  = w["FPOGD"].mean()     if "FPOGD"  in w else np.nan
        fix_aoi_ratio = w["AOI"].notna().mean() if "AOI" in w else np.nan
        fix_efficiency = fix_count / (abs(fix_mean_dur) + eps) if pd.notna(fix_mean_dur) else np.nan
        return pd.Series({
            "fix_efficiency": fix_efficiency,
            "fix_aoi_ratio": fix_aoi_ratio,
        })
    return (fix_df.dropna(subset=["win_id"])
                 .groupby(BASE_KEYS, dropna=False)
                 .apply(_agg)
                 .reset_index())

def window_features_gaze(gaze_df: pd.DataFrame, *, win_sec: float = 3.0) -> pd.DataFrame:
    def _agg(w: pd.DataFrame) -> pd.Series:
        gv = (w["FPOGV"] == 1) if "FPOGV" in w else pd.Series(True, index=w.index)
        gx = w.loc[gv, "FPOGX"].std() if "FPOGX" in w else np.nan
        gy = w.loc[gv, "FPOGY"].std() if "FPOGY" in w else np.nan
        gaze_disp_total = np.sqrt((0 if pd.isna(gx) else gx**2) + (0 if pd.isna(gy) else gy**2)) if (pd.notna(gx) or pd.notna(gy)) else np.nan
        if "LPMM" in w and "RPMM" in w:
            l = w["LPMM"].where(w.get("LPMMV", 1)==1)
            r = w["RPMM"].where(w.get("RPMMV", 1)==1)
            pupil_row = pd.concat([l, r], axis=1).mean(axis=1, skipna=True)
            pupil_mean = pupil_row.mean()
            pupil_var  = pupil_row.var(ddof=1)
        else:
            pupil_mean, pupil_var = np.nan, np.nan
        saccade_mean_mag = w.loc[gv, "SACCADE_MAG"].mean() if "SACCADE_MAG" in w else np.nan
        saccade_dir_var  = w.loc[gv, "SACCADE_DIR"].var(ddof=1) if "SACCADE_DIR" in w else np.nan
        blink_count = w.loc[gv, "BKID"].nunique() if "BKID" in w else np.nan
        blink_rate  = (blink_count / win_sec) if pd.notna(blink_count) else np.nan
        valid_gaze_ratio = gv.mean() if len(w) else np.nan
        return pd.Series({
            "gaze_dispersion_total": gaze_disp_total,
            "pupil_mean": pupil_mean,
            "pupil_var": 0 if pd.isna(pupil_var) else pupil_var,
            "saccade_mean_mag": saccade_mean_mag,
            "saccade_dir_var": 0 if pd.isna(saccade_dir_var) else saccade_dir_var,
            "blink_rate": blink_rate,
            "valid_gaze_ratio": valid_gaze_ratio,
        })
    return (gaze_df.dropna(subset=["win_id"])
                 .groupby(BASE_KEYS, dropna=False)
                 .apply(_agg)
                 .reset_index())

In [33]:
def compute_on_fraction(gaze_with_win: pd.DataFrame) -> pd.DataFrame:
    """
    Returns one row per window with:
      - on_fraction: fraction of rows labeled ON in that window
      - n_rows, n_on, n_off (for diagnostics)
    Uses simple sample-count fraction (robust + fast).
    """
    g = gaze_with_win.copy()
    g["label"] = g["label"].astype(str).str.upper().str.strip()
    g = g[g["label"].isin(["ON","OFF"])]
    # one-hot ON
    g["is_on"] = (g["label"] == "ON").astype(int)
    agg = (g.dropna(subset=["win_id"])
             .groupby(BASE_KEYS, dropna=False)
             .agg(n_rows=("is_on","size"),
                  n_on=("is_on","sum"))
             .reset_index())
    agg["n_off"] = agg["n_rows"] - agg["n_on"]
    agg["on_fraction"] = agg["n_on"] / agg["n_rows"].where(agg["n_rows"]>0, np.nan)
    return agg

In [34]:
def relabel_by_fraction(windows_df: pd.DataFrame,
                        frac_df: pd.DataFrame,
                        on_hi: float = 0.80,
                        off_lo: float = 0.20) -> pd.DataFrame:
    """
    Join on_fraction → assign clean labels: 'ON' if ≥on_hi, 'OFF' if ≤off_lo, else drop.
    Prints retention stats.
    """
    win = windows_df.merge(frac_df[BASE_KEYS + ["on_fraction","n_rows","n_on","n_off"]],
                           on=BASE_KEYS, how="left", validate="one_to_one")
    before = len(win)
    # assign labels
    win["label"] = np.where(win["on_fraction"] >= on_hi, "ON",
                           np.where(win["on_fraction"] <= off_lo, "OFF", np.nan))
    kept = win["label"].notna().sum()
    print(f"QC by on_fraction: kept {kept}/{before} windows ({kept/before:.1%}) "
          f"— thresholds: ON≥{on_hi}, OFF≤{off_lo}")
    # optional: show class balance after QC
    print(win["label"].value_counts(dropna=True))
    return win.loc[win["label"].notna()].copy()

In [35]:
# Example for one split (say, Task 1.x ON/OFF raws already loaded as fix_df, gaze_df):

# 1) Align windows
anchor   = build_shared_anchor(fix_df, gaze_df)
fix_wid  = add_window_id_by_timetick(fix_df,  win_sec=3.0, shared_ref=anchor)
gaze_wid = add_window_id_by_timetick(gaze_df, win_sec=3.0, shared_ref=anchor)

# 2) Features (now grouped WITHOUT label)
fix_w = window_features_fixations(fix_wid)
gaz_w = window_features_gaze(gaze_wid, win_sec=3.0)

# 3) Merge features
windows = (fix_w.merge(gaz_w, on=BASE_KEYS, how="outer", validate="one_to_one")
                .sort_values(BASE_KEYS)
                .reset_index(drop=True))

# 4) Compute on_fraction from the raw gaze rows + window ids
frac = compute_on_fraction(gaze_wid)

# 5) Relabel with thresholds and filter
windows_qc = relabel_by_fraction(windows, frac, on_hi=0.80, off_lo=0.20)

# 6) Save and use windows_qc for PCA / clustering / supervised CV
windows_qc.to_csv("outputs/windows_task1_experiment.csv", index=False)

/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_24838/4232109533.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_24838/4232109533.py:50: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)


QC by on_fraction: kept 19964/19964 windows (100.0%) — thresholds: ON≥0.8, OFF≤0.2
label
nan    11943
ON      4842
OFF     3179
Name: count, dtype: int64
